In [0]:
# Objective: Recover and interpret the treatment effect with
# full statistical rigor.
#
# Ground truth (from Step 5): injected lift = +4pp
# We expect to recover approximately this value.
# All results are from a simulated experiment — stated plainly.

GOLD = "churn_project.gold"

import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings("ignore")

# Pull experiment results to pandas — eligible cohort only, manageable size
df = spark.table(f"{GOLD}.experiment_results").toPandas()

print(f"Experiment results loaded: {df.shape}")
print(f"\nGroup counts:")
print(df["group"].value_counts())
print(f"\nColumn dtypes:")
print(df.dtypes)


Experiment results loaded: (205883, 11)

Group counts:
group
control      103138
treatment    102745
Name: count, dtype: int64

Column dtypes:
msno                          object
group                         object
renewed                        int64
revenue                      float64
churn_probability            float64
risk_segment                  object
auto_renew_at_checkpoint       int32
ever_cancelled                 int32
engagement_bucket             object
tenure_bucket                 object
amount_paid_at_checkpoint      int32
dtype: object


In [0]:
from scipy.stats import chisquare

counts   = df["group"].value_counts()
treat_n  = counts["treatment"]
ctrl_n   = counts["control"]
total_n  = len(df)

observed = [treat_n, ctrl_n]
expected = [total_n / 2, total_n / 2]
chi2, p  = chisquare(f_obs=observed, f_exp=expected)

print("=" * 55)
print("SAMPLE RATIO MISMATCH (SRM) CHECK")
print("=" * 55)
print(f"  Treatment : {treat_n:,}  ({treat_n/total_n*100:.2f}%)")
print(f"  Control   : {ctrl_n:,}  ({ctrl_n/total_n*100:.2f}%)")
print(f"  Chi-square: {chi2:.4f}")
print(f"  p-value   : {p:.4f}")
print(f"\n  Result    : {'✓ No SRM — assignment held' if p > 0.05 else '✗ SRM detected'}")
print(f"""
Note: with self-controlled randomization this check will always
pass. In a production system, SRM catches logging bugs, redirect
failures, or pipeline errors that break assignment
""")

SAMPLE RATIO MISMATCH (SRM) CHECK
  Treatment : 102,745  (49.90%)
  Control   : 103,138  (50.10%)
  Chi-square: 0.7502
  p-value   : 0.3864

  Result    : ✓ No SRM — assignment held

Note: with self-controlled randomization this check will always
pass. In a production system, SRM catches logging bugs, redirect
failures, or pipeline errors that break assignment



In [0]:
# Core renewal rates
ctrl_renewed  = df[df["group"] == "control"]["renewed"].sum()
treat_renewed = df[df["group"] == "treatment"]["renewed"].sum()
ctrl_rate     = ctrl_renewed / ctrl_n
treat_rate    = treat_renewed / treat_n
abs_lift      = treat_rate - ctrl_rate
rel_lift      = abs_lift / ctrl_rate

print("=" * 55)
print("LIFT ANALYSIS")
print("=" * 55)
print(f"  Control   : {ctrl_renewed:,} / {ctrl_n:,} = {ctrl_rate*100:.4f}%")
print(f"  Treatment : {treat_renewed:,} / {treat_n:,} = {treat_rate*100:.4f}%")
print(f"\n  Absolute lift : +{abs_lift*100:.4f}pp")
print(f"  Relative lift : +{rel_lift*100:.2f}%")
print(f"\n  Ground truth (injected) : +12.00pp")
print(f"  Recovered estimate      : +{abs_lift*100:.2f}pp")
print(f"  Difference from truth   : {(abs_lift - 0.04)*100:.4f}pp")

LIFT ANALYSIS
  Control   : 60,956 / 103,138 = 59.1014%
  Treatment : 73,041 / 102,745 = 71.0896%

  Absolute lift : +11.9882pp
  Relative lift : +20.28%

  Ground truth (injected) : +12.00pp
  Recovered estimate      : +11.99pp
  Difference from truth   : 7.9882pp


In [0]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

# Two-proportion z-test
count_arr = np.array([treat_renewed, ctrl_renewed])
nobs_arr  = np.array([treat_n, ctrl_n])

z_stat, p_value = proportions_ztest(count_arr, nobs_arr, alternative="two-sided")

# 95% CI on the absolute lift (treatment - control)
ci_low, ci_high = confint_proportions_2indep(
    treat_renewed, treat_n,
    ctrl_renewed,  ctrl_n,
    method="newcomb"
)

print("=" * 55)
print("HYPOTHESIS TEST — Two-proportion z-test")
print("=" * 55)
print(f"  H0: renewal rate is equal in treatment and control")
print(f"  H1: renewal rates differ (two-sided)")
print(f"\n  z-statistic : {z_stat:.4f}")
print(f"  p-value     : {p_value:.6f}")
print(f"\n  95% CI on absolute lift: [{ci_low*100:.4f}pp, {ci_high*100:.4f}pp]")
print(f"\n  Result: {'✓ SIGNIFICANT — reject H0' if p_value < 0.05 else '✗ NOT SIGNIFICANT'} at alpha=0.05")
print(f"""
Interpretation: the 95% CI on the absolute lift is
[{ci_low*100:.2f}pp, {ci_high*100:.2f}pp]. We lead with the CI rather
than just the p-value — the CI tells us both the direction and
the plausible magnitude of the effect, not just whether it exists.
""")

HYPOTHESIS TEST — Two-proportion z-test
  H0: renewal rate is equal in treatment and control
  H1: renewal rates differ (two-sided)

  z-statistic : 57.0537
  p-value     : 0.000000

  95% CI on absolute lift: [11.5794pp, 12.3964pp]

  Result: ✓ SIGNIFICANT — reject H0 at alpha=0.05

Interpretation: the 95% CI on the absolute lift is
[11.58pp, 12.40pp]. We lead with the CI rather
than just the p-value — the CI tells us both the direction and
the plausible magnitude of the effect, not just whether it exists.



In [0]:
# Revenue per user by group
ctrl_rpu   = df[df["group"] == "control"]["revenue"].mean()
treat_rpu  = df[df["group"] == "treatment"]["revenue"].mean()

# Incremental renewals: how many MORE renewals did treatment produce?
# vs what we'd expect if they'd renewed at the control rate
expected_treat_renewals = treat_n * ctrl_rate
actual_treat_renewals   = treat_renewed
incremental_renewals    = actual_treat_renewals - expected_treat_renewals

# Discount cost: total revenue given away to ALL treatment renewers
# (both incremental AND would-have-renewed-anyway users)
avg_full_price = df[df["group"] == "control"]["amount_paid_at_checkpoint"].mean()
total_treat_renewals = treat_renewed
discount_per_renewer = avg_full_price * 0.10
total_discount_cost  = total_treat_renewals * discount_per_renewer

# Cannibalization cost: discount given specifically to would-have-renewed users
# (these users represent pure revenue loss — no retention benefit)
would_have_renewed   = expected_treat_renewals
cannibalization_cost = would_have_renewed * discount_per_renewer

# Incremental revenue: revenue from the EXTRA renewals the discount produced
incremental_revenue  = incremental_renewals * avg_full_price * (1 - 0.10)

# Net business impact
net_impact = incremental_revenue - cannibalization_cost

print("=" * 60)
print("REVENUE ANALYSIS")
print("=" * 60)
print(f"\n  Average full price (control group)    : {avg_full_price:.2f} NTD")
print(f"  Discount per treatment renewer (30%)  : {discount_per_renewer:.2f} NTD")

print(f"\n── Renewal counts ──")
print(f"  Control renewals                      : {ctrl_renewed:,}")
print(f"  Treatment renewals (actual)           : {treat_renewed:,}")
print(f"  Expected treatment renewals           : {expected_treat_renewals:,.0f}  (at control rate)")
print(f"  Incremental renewals from discount    : {incremental_renewals:,.0f}")

print(f"\n── Revenue per user ──")
print(f"  Control RPU                           : {ctrl_rpu:.2f} NTD")
print(f"  Treatment RPU                         : {treat_rpu:.2f} NTD")
print(f"  RPU gap (discount cost visible)       : {treat_rpu - ctrl_rpu:.2f} NTD")

print(f"\n── Business impact ──")
print(f"  Total discount cost (all renewers)    : {total_discount_cost:>12,.0f} NTD")
print(f"  Cannibalization cost (wasted discount): {cannibalization_cost:>12,.0f} NTD")
print(f"  Incremental revenue (new renewals)    : {incremental_revenue:>12,.0f} NTD")
print(f"  Net business impact                   : {net_impact:>12,.0f} NTD")
print(f"\n  Result: {'✓ NET POSITIVE' if net_impact > 0 else '✗ NET NEGATIVE'}")

print(f"\n── Efficiency metric ──")
discount_per_incremental = total_discount_cost / incremental_renewals
print(f"  Total discount cost per incremental   ")
print(f"  renewal                               : {discount_per_incremental:.2f} NTD")
print(f"  (vs avg full price of {avg_full_price:.2f} NTD)")
print(f"  {'✓ Cost per incremental renewal < full price' if discount_per_incremental < avg_full_price else '✗ Cost exceeds full price'}")

REVENUE ANALYSIS

  Average full price (control group)    : 250.20 NTD
  Discount per treatment renewer (30%)  : 25.02 NTD

── Renewal counts ──
  Control renewals                      : 60,956
  Treatment renewals (actual)           : 73,041
  Expected treatment renewals           : 60,724  (at control rate)
  Incremental renewals from discount    : 12,317

── Revenue per user ──
  Control RPU                           : 147.98 NTD
  Treatment RPU                         : 160.18 NTD
  RPU gap (discount cost visible)       : 12.20 NTD

── Business impact ──
  Total discount cost (all renewers)    :    1,827,465 NTD
  Cannibalization cost (wasted discount):    1,519,290 NTD
  Incremental revenue (new renewals)    :    2,773,571 NTD
  Net business impact                   :    1,254,280 NTD

  Result: ✓ NET POSITIVE

── Efficiency metric ──
  Total discount cost per incremental   
  renewal                               : 148.37 NTD
  (vs avg full price of 250.20 NTD)
  ✓ Cost per incre

In [0]:
# Pre-specified segments (as stated in the experiment design doc):
# 1. Auto-renew status
# 2. Cancellation history
# 3. Engagement bucket
# 4. Tenure bucket
# 5. Churn probability sub-band within High segment

def segment_analysis(df, segment_col, label=None):
    """
    Compute lift, z-test, and net revenue impact for each level
    of a given segment column.
    """
    label = label or segment_col
    results = []

    for seg_val in sorted(df[segment_col].dropna().unique()):
        seg_df = df[df[segment_col] == seg_val]
        ctrl   = seg_df[seg_df["group"] == "control"]
        treat  = seg_df[seg_df["group"] == "treatment"]

        ctrl_n_s    = len(ctrl)
        treat_n_s   = len(treat)
        ctrl_ren_s  = ctrl["renewed"].sum()
        treat_ren_s = treat["renewed"].sum()

        if ctrl_n_s == 0 or treat_n_s == 0:
            continue

        ctrl_rate_s  = ctrl_ren_s / ctrl_n_s
        treat_rate_s = treat_ren_s / treat_n_s
        abs_lift_s   = treat_rate_s - ctrl_rate_s

        # z-test
        z_s, p_s = proportions_ztest(
            [treat_ren_s, ctrl_ren_s],
            [treat_n_s, ctrl_n_s],
            alternative="two-sided"
        )

        # Revenue impact
        avg_price_s          = ctrl["amount_paid_at_checkpoint"].mean()
        expected_treat_ren_s = treat_n_s * ctrl_rate_s
        incremental_ren_s    = treat_ren_s - expected_treat_ren_s
        cannib_cost_s        = expected_treat_ren_s * avg_price_s * 0.10
        incr_rev_s           = incremental_ren_s * avg_price_s * 0.90
        net_impact_s         = incr_rev_s - cannib_cost_s

        results.append({
            "segment":            f"{label}={seg_val}",
            "ctrl_n":             ctrl_n_s,
            "treat_n":            treat_n_s,
            "ctrl_renewal_rate":  round(ctrl_rate_s * 100, 2),
            "treat_renewal_rate": round(treat_rate_s * 100, 2),
            "abs_lift_pp":        round(abs_lift_s * 100, 2),
            "p_value":            round(p_s, 4),
            "significant":        "✓" if p_s < 0.05 else "✗",
            "net_impact_NTD":     round(net_impact_s, 0),
            "net_positive":       "✓" if net_impact_s > 0 else "✗",
        })

    return pd.DataFrame(results)

In [0]:
print("=" * 70)
print("SEGMENT ANALYSIS — Pre-specified, exploratory")
print("Note: multiple comparisons — no individual result drives the decision")
print("=" * 70)

# 1. Auto-renew status
print("\n── 1. Auto-renew status ──")
seg_autorenew = segment_analysis(df, "auto_renew_at_checkpoint", "auto_renew")
print(seg_autorenew.to_string(index=False))

# 2. Cancellation history
print("\n── 2. Cancellation history ──")
seg_cancel = segment_analysis(df, "ever_cancelled", "ever_cancelled")
print(seg_cancel.to_string(index=False))

# 3. Engagement bucket
print("\n── 3. Engagement bucket ──")
seg_engagement = segment_analysis(df, "engagement_bucket", "engagement")
print(seg_engagement.to_string(index=False))

# 4. Tenure bucket
print("\n── 4. Tenure bucket ──")
seg_tenure = segment_analysis(
    df[df["tenure_bucket"].notna()], "tenure_bucket", "tenure"
)
print(seg_tenure.to_string(index=False))

# 5. Churn probability sub-band
print("\n── 5. Churn probability sub-band within High segment ──")
df["prob_band"] = pd.cut(
    df["churn_probability"],
    bins=[0.556, 0.70, 0.90, 1.001],
    labels=["0.556-0.70 (borderline)", "0.70-0.90 (confident)", "0.90-1.0 (near-certain)"]
)
seg_prob = segment_analysis(df, "prob_band", "prob_band")
print(seg_prob.to_string(index=False))

SEGMENT ANALYSIS — Pre-specified, exploratory
Note: multiple comparisons — no individual result drives the decision

── 1. Auto-renew status ──
     segment  ctrl_n  treat_n  ctrl_renewal_rate  treat_renewal_rate  abs_lift_pp  p_value significant  net_impact_NTD net_positive
auto_renew=0   54060    53925               59.1               70.85        11.75      0.0           ✓        845877.0            ✓
auto_renew=1   49078    48820               59.1               71.35        12.25      0.0           ✓        388642.0            ✓

── 2. Cancellation history ──
         segment  ctrl_n  treat_n  ctrl_renewal_rate  treat_renewal_rate  abs_lift_pp  p_value significant  net_impact_NTD net_positive
ever_cancelled=0   97838    97476              59.14               71.04        11.90      0.0           ✓       1196356.0            ✓
ever_cancelled=1    5300     5269              58.32               71.93        13.61      0.0           ✓         48964.0            ✓

── 3. Engagement buc

## Segment Analysis Findings

All five pre-specified segment breakdowns show statistically significant
lift and net positive revenue impact at 10% discount / 12pp injected lift.
Results are reported as exploratory — multiple comparisons apply.

**Key observations:**

1. **Lift is uniform across segments (11.5–13.6pp range)** — consistent
   with the model calibration finding: within the High-risk segment, the
   model has no discriminating power, so no sub-group responds dramatically
   differently to the discount than the overall population.

2. **Cancellers show highest lift (13.61pp)** — the 5,269-user segment of
   users who actively cancelled at least once responds most strongly,
   suggesting the discount tips users who are reconsidering back toward
   renewal. Small segment but worth prioritizing.

3. **New users (<6 months) show second-highest lift (12.70pp)** — early
   in the subscription lifecycle, a discount during the first renewal
   may establish a retention habit. Consider prioritizing new users in
   future targeting.

4. **Churn probability sub-band shows no meaningful variation** — lift
   is nearly identical across borderline (12.01pp), confident (11.73pp),
   and near-certain (12.08pp) sub-bands, confirming flat model calibration
   within the High segment. Finer probability-based targeting adds no
   value with the current model.

5. **Engagement and tenure show no meaningful lift variation** — consistent
   with EDA finding that neither predicts churn; neither moderates the
   discount's effectiveness either.

**Multiple comparisons note:** with 14 segment-level tests, approximately
0-1 false positives would be expected at alpha=0.05 under the null. All
segments show p≈0.000, so the lift findings are robust. However, the
*magnitude* differences between segments (e.g. 13.61pp vs 11.51pp) should
not be treated as precise estimates — they reflect noise as much as signal
at these sub-group sizes.

In [0]:
print("=" * 65)
print("EXPERIMENT ANALYSIS — FINAL SUMMARY")
print("=" * 65)
print(f"""
PARAMETERS
──────────
  Discount rate          : 10% (justified by sensitivity analysis)
  Injected lift          : 12pp (true effect, unknown to analyst)
  Design MDE             : 10pp (experiment was powered for this)
  Baseline renewal rate  : 59.14% (high-risk proxy, training cohort)

PRIMARY RESULT
──────────────
  Control renewal rate   : {ctrl_rate*100:.2f}%
  Treatment renewal rate : {treat_rate*100:.2f}%
  Recovered lift         : +{abs_lift*100:.2f}pp
  95% CI                 : [{ci_low*100:.2f}pp, {ci_high*100:.2f}pp]
  p-value                : < 0.0001
  vs injected truth      : +12pp  (recovered {abs_lift*100:.2f}pp — within noise ✓)
  vs design MDE          : CI lower bound {ci_low*100:.2f}pp > 10pp MDE ✓

REVENUE RESULT
──────────────
  Control RPU            : {ctrl_rpu:.2f} NTD
  Treatment RPU          : {treat_rpu:.2f} NTD
  Incremental renewals   : {incremental_renewals:,.0f}
  Cannibalization cost   : {cannibalization_cost:,.0f} NTD
  Incremental revenue    : {incremental_revenue:,.0f} NTD
  Net business impact    : +{net_impact:,.0f} NTD  ✓ NET POSITIVE

SEGMENT RESULT
──────────────
  All 14 segment-level tests: significant lift ✓ and net positive ✓
  Highest lift    : ever_cancelled=1  (13.61pp)
  Highest n.impact: prob_band 0.90+   (+807,236 NTD, largest segment)
  Uniform pattern : lift varies only 2pp across all segments —
                    no strong targeting signal within the High group

DECISION
────────
  ✓ GO — statistically significant lift exceeding the 10pp MDE,
    net positive revenue impact across all segments,
    discount rate justified by pre-experiment sensitivity analysis.

  Targeting recommendation → Step 8 business recommendation.
""")

EXPERIMENT ANALYSIS — FINAL SUMMARY

PARAMETERS
──────────
  Discount rate          : 10% (justified by sensitivity analysis)
  Injected lift          : 12pp (true effect, unknown to analyst)
  Design MDE             : 10pp (experiment was powered for this)
  Baseline renewal rate  : 59.14% (high-risk proxy, training cohort)

PRIMARY RESULT
──────────────
  Control renewal rate   : 59.10%
  Treatment renewal rate : 71.09%
  Recovered lift         : +11.99pp
  95% CI                 : [11.58pp, 12.40pp]
  p-value                : < 0.0001
  vs injected truth      : +12pp  (recovered 11.99pp — within noise ✓)
  vs design MDE          : CI lower bound 11.58pp > 10pp MDE ✓

REVENUE RESULT
──────────────
  Control RPU            : 147.98 NTD
  Treatment RPU          : 160.18 NTD
  Incremental renewals   : 12,317
  Cannibalization cost   : 1,519,290 NTD
  Incremental revenue    : 2,773,571 NTD
  Net business impact    : +1,254,280 NTD  ✓ NET POSITIVE

SEGMENT RESULT
──────────────
  All 14 s